[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavinciDreams/SymbioGPT/blob/main/distill_test.ipynb)

# Distillation A/B Test: Does Teacher Guidance Help?

Trains two identical ~5M student architectures:
- **Student A (Distilled)**: KL divergence from SymbioGPT-10M teacher + CE loss
- **Student B (From Scratch)**: CE loss only

Student architecture is informed by teacher gate weights:
- Layer 0: conv only (teacher gate: 56% conv)
- Layers 1-5: conv + attention (teacher gate: attention dominates)
- No monarch/longconv (weakest organelles at 15-20%)

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup
!pip install -q wandb huggingface_hub
!git clone https://github.com/DavinciDreams/SymbioGPT.git /content/SymbioGPT
%cd /content/SymbioGPT

In [ ]:
# 2. GPU check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# 3. W&B + HF login
import wandb
from huggingface_hub import login as hf_login

wandb.login()
hf_login()

In [ ]:
# 4. Load data + teacher from HuggingFace
import os
from huggingface_hub import hf_hub_download

HF_REPO = "LisaMegaWatts/SymbioGPT-10M"
os.makedirs("data", exist_ok=True)

print("Downloading training tokens...")
hf_hub_download(repo_id=HF_REPO, filename="data/train_curated.txt.tokens.pt", local_dir=".")
print("Downloading validation tokens...")
hf_hub_download(repo_id=HF_REPO, filename="data/val.txt.tokens.pt", local_dir=".")
print("Downloading teacher model...")
hf_hub_download(repo_id=HF_REPO, filename="symbio_best.pt", local_dir=".")
print("All files downloaded.")

In [ ]:
# 5. Load model code and prepare data
import subprocess, sys, importlib
subprocess.run(["git", "pull"], cwd="/content/SymbioGPT", check=True)
sys.path.insert(0, "/content/SymbioGPT")

import symbio_model
importlib.reload(symbio_model)
from symbio_model import (
    SymbioConfig, SymbioGPT, compute_symbio_params,
    complexity_penalty, compute_gate_entropy,
)

# Load and chunk tokens
CTX = 256
print("Loading tokens...")
train_tokens = torch.load("data/train_curated.txt.tokens.pt", weights_only=True).tolist()
val_tokens = torch.load("data/val.txt.tokens.pt", weights_only=True).tolist()

def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f"Train: {len(train_inputs):,} seqs ({len(train_inputs)*CTX:,} tokens)")
print(f"Val: {len(val_inputs):,} seqs")
del train_tokens, val_tokens

In [ ]:
# 6. Load teacher model (frozen)
teacher_config = SymbioConfig(
    d_model=320, n_layers=8, n_heads=5, head_dim=64, ffn_mult=4,
    context_length=256, vocab_size=2000, weight_tying=True,
    organelles=("causal_conv", "monarch", "long_conv", "attention"),
    conv_kernel_size=4, n_monarch_heads=1,
    gate_temperature_init=1.0, free_energy_beta=0.001,
)

device = torch.device("cuda")
teacher = SymbioGPT(teacher_config).to(device)

# Load trained weights
state_dict = torch.load("symbio_best.pt", map_location=device, weights_only=True)
teacher.load_state_dict(state_dict)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f"Teacher loaded: {teacher_params:,} params (frozen)")

# Quick sanity check
with torch.no_grad():
    test_out = teacher(train_inputs[:2].to(device))
print(f"Teacher forward OK: {test_out.shape}")

In [ ]:
# 7. Define student config (~5M params, guided by teacher gate weights)
student_config = SymbioConfig(
    d_model=256,
    n_layers=6,
    n_heads=4,
    head_dim=64,
    ffn_mult=4,
    context_length=256,
    vocab_size=2000,
    weight_tying=True,
    organelles=("causal_conv", "attention"),
    conv_kernel_size=4,
    n_monarch_heads=1,
    gate_temperature_init=1.0,
    free_energy_beta=0.0,  # No complexity penalty for students
    per_layer_organelles=[
        ("causal_conv",),                    # Layer 0: conv only
        ("causal_conv", "attention"),         # Layers 1-5: conv + attention
        ("causal_conv", "attention"),
        ("causal_conv", "attention"),
        ("causal_conv", "attention"),
        ("causal_conv", "attention"),
    ],
)

# Verify
test_student = SymbioGPT(student_config)
student_params = sum(p.numel() for p in test_student.parameters())
print(f"Student config: d={student_config.d_model}, L={student_config.n_layers}")
print(f"Student params: {student_params:,}")
print(f"Teacher params: {teacher_params:,}")
print(f"Ratio: {teacher_params/student_params:.1f}x")
print(f"\nPer-layer organelles:")
for i, orgs in enumerate(student_config.per_layer_organelles):
    print(f"  Layer {i}: {', '.join(orgs)}")
del test_student

In [ ]:
# 8. Shared training function
import math
import time
import torch.nn.functional as F

def evaluate(model, val_inputs, val_labels, batch_size, device, amp_dtype):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for i in range(0, len(val_inputs), batch_size):
            batch_in = val_inputs[i:i+batch_size].to(device)
            batch_tgt = val_labels[i:i+batch_size].to(device)
            with torch.amp.autocast("cuda", enabled=amp_dtype is not None, dtype=amp_dtype):
                logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction="sum"
            )
            total_loss += loss.item()
            total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl


def train_student(config, train_inputs, train_labels, val_inputs, val_labels,
                  teacher=None, alpha=0.5, temperature=2.0,
                  steps=5000, batch_size=64, lr=6e-4,
                  eval_interval=250, run_name="student", seed=42):
    """Train a student, optionally with distillation. Returns val history."""
    torch.manual_seed(seed)

    device = torch.device("cuda")
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

    model = SymbioGPT(config).to(device)
    if hasattr(torch, "compile"):
        model = torch.compile(model)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=0.1, betas=(0.9, 0.95)
    )
    warmup_steps = 300

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(steps - warmup_steps, 1)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    mode = "distilled" if teacher is not None else "scratch"
    run = wandb.init(
        project="symbiogenesis",
        name=f"{run_name}-{mode}",
        config={
            "mode": mode,
            "student_params": sum(p.numel() for p in model.parameters()),
            "d_model": config.d_model,
            "n_layers": config.n_layers,
            "steps": steps,
            "batch_size": batch_size,
            "lr": lr,
            "alpha": alpha if teacher else 1.0,
            "temperature": temperature if teacher else 0,
            "seed": seed,
        },
        tags=["distillation-test", mode, "student"],
        reinit=True,
    )

    n_train = len(train_inputs)
    history = []
    best_val_loss = float("inf")
    t_start = time.time()
    step = 0

    print(f"\n{'='*60}")
    print(f"Training: {run_name} ({mode})")
    print(f"{'='*60}")

    model.train()
    while step < steps:
        perm = torch.randperm(n_train, generator=torch.Generator().manual_seed(seed + step // n_train))
        for i in range(0, n_train, batch_size):
            if step >= steps:
                break

            idx = perm[i:i+batch_size]
            batch_in = train_inputs[idx].to(device)
            batch_tgt = train_labels[idx].to(device)

            with torch.amp.autocast("cuda", enabled=True, dtype=amp_dtype):
                student_logits = model(batch_in)
                B, T, V = student_logits.shape

                ce_loss = F.cross_entropy(
                    student_logits.reshape(B*T, V), batch_tgt.reshape(B*T)
                )

                if teacher is not None:
                    with torch.no_grad():
                        teacher_logits = teacher(batch_in)
                    kd_loss = F.kl_div(
                        F.log_softmax(student_logits.reshape(B*T, V) / temperature, dim=-1),
                        F.softmax(teacher_logits.reshape(B*T, V) / temperature, dim=-1),
                        reduction="batchmean",
                    ) * (temperature ** 2)
                    loss = alpha * ce_loss + (1.0 - alpha) * kd_loss
                else:
                    kd_loss = torch.tensor(0.0)
                    loss = ce_loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

            # Log
            if step % 50 == 0:
                wandb.log({
                    "train/ce_loss": ce_loss.item(),
                    "train/kd_loss": kd_loss.item() if teacher else 0,
                    "train/total_loss": loss.item(),
                    "train/lr": scheduler.get_last_lr()[0],
                }, step=step)

            # Eval
            if step > 0 and step % eval_interval == 0:
                val_loss, val_ppl = evaluate(
                    model, val_inputs, val_labels, batch_size, device, amp_dtype
                )
                wandb.log({"val/loss": val_loss, "val/perplexity": val_ppl}, step=step)
                history.append((step, val_loss, val_ppl))
                marker = " ** BEST **" if val_loss < best_val_loss else ""
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                elapsed = time.time() - t_start
                print(f"  [{mode} step {step:5d}] val_loss={val_loss:.4f} ppl={val_ppl:.1f} "
                      f"CE={ce_loss.item():.4f} elapsed={elapsed:.0f}s{marker}")
                model.train()

            step += 1

    # Final eval
    val_loss, val_ppl = evaluate(model, val_inputs, val_labels, batch_size, device, amp_dtype)
    history.append((steps, val_loss, val_ppl))
    wandb.log({"val/final_loss": val_loss, "val/final_ppl": val_ppl})
    wandb.finish()

    elapsed = time.time() - t_start
    print(f"  {mode} FINAL: val_loss={val_loss:.4f} ppl={val_ppl:.1f} ({elapsed:.0f}s)")

    return model, history

In [ ]:
# 9. Train Student A — Distilled from teacher
student_a, history_a = train_student(
    student_config, train_inputs, train_labels, val_inputs, val_labels,
    teacher=teacher, alpha=0.5, temperature=2.0,
    steps=5000, batch_size=64, lr=6e-4,
    run_name="distill-test", seed=42,
)

In [ ]:
# 10. Train Student B — From scratch (no teacher)
student_b, history_b = train_student(
    student_config, train_inputs, train_labels, val_inputs, val_labels,
    teacher=None,
    steps=5000, batch_size=64, lr=6e-4,
    run_name="distill-test", seed=42,
)

In [ ]:
# 11. Compare results
print("\n" + "="*60)
print("DISTILLATION A/B TEST RESULTS")
print("="*60)
print(f"\nStudent: {sum(p.numel() for p in student_a.parameters()):,} params")
print(f"Teacher: {teacher_params:,} params (PPL 35.3)")
print(f"\n{'Step':>6} | {'Distilled PPL':>14} | {'Scratch PPL':>12} | {'Delta':>8}")
print("-" * 50)

for (step_a, _, ppl_a), (step_b, _, ppl_b) in zip(history_a, history_b):
    delta = ppl_b - ppl_a
    winner = "<- distill" if delta > 0 else "<- scratch"
    print(f"{step_a:6d} | {ppl_a:14.1f} | {ppl_b:12.1f} | {delta:+8.1f} {winner}")

# Final summary
final_a = history_a[-1][2]
final_b = history_b[-1][2]
delta = final_b - final_a
print(f"\nFinal PPL: Distilled={final_a:.1f}, Scratch={final_b:.1f}, Delta={delta:+.1f}")

if delta > 0:
    print(f"\n>>> DISTILLATION WINS by {delta:.1f} PPL points <<<")
    print("Teacher soft targets provide meaningful guidance.")
    print("Proceed with evolutionary architecture search using distillation.")
elif delta < -1:
    print(f"\n>>> SCRATCH WINS by {-delta:.1f} PPL points <<<")
    print("Teacher may not add enough signal for this architecture gap.")
    print("Consider: larger teacher, longer training, or different alpha/temperature.")
else:
    print(f"\n>>> ROUGHLY TIED (delta={delta:+.1f}) <<<")
    print("Distillation neither helps nor hurts significantly.")

In [ ]:
# 12. Upload winner to HuggingFace
from huggingface_hub import HfApi

hf_api = HfApi()
winner = student_a if final_a <= final_b else student_b
winner_mode = "distilled" if final_a <= final_b else "scratch"
winner_ppl = min(final_a, final_b)

save_path = "checkpoints/distill_test_winner.pt"
os.makedirs("checkpoints", exist_ok=True)
torch.save(winner.state_dict(), save_path)

try:
    hf_api.upload_file(
        path_or_fileobj=save_path,
        path_in_repo=f"distill_test/student_{winner_mode}.pt",
        repo_id=HF_REPO,
        commit_message=f"Distillation test winner ({winner_mode}, PPL={winner_ppl:.1f}, 5M params)",
    )
    print(f"Uploaded to HF: distill_test/student_{winner_mode}.pt")
except Exception as e:
    print(f"HF upload failed: {e}")

print("\nDone!")